# get-children-callable-param — faded example 1: Fill the MiniTensor filter in get_children

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `get-children-callable-param`. The last cell reports your progress on the `Backprop: get_children callable param` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: get_children callable param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`get-children-callable-param`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "get-children-callable-param"
DD_SUBTOPIC = "Backprop: get_children callable param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`get_children` yields `(name, value)` for every attribute in `self.__dict__` that is a `MiniTensor`, skipping config attributes. The `isinstance` guard is what separates trainable state from configuration.

## Faded exercise 1

Implement `Module.get_children` as a generator. Scan `self.__dict__.items()` and yield `(name, val)` only for attributes that are `MiniTensor` instances. Complete the blanked filter condition.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(3)

class MiniTensor:
    def __init__(self, data):
        self.data = data

class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            keep = isinstance(val, MiniTensor)
            if keep:
                yield name, val

class Linear(Module):
    def __init__(self):
        self.weight = MiniTensor(t.randn(4, 3))
        self.bias = MiniTensor(t.zeros(4))
        self.in_features = 3

print([n for n, _ in Linear().get_children()])


def _test():
    lin = Linear()
    children = list(lin.get_children())
    names = [n for n, _ in children]
    # independent expectation: only the two tensor attributes, config int excluded
    assert names == ['weight', 'bias'], names
    for _, v in children:
        assert isinstance(v, MiniTensor)
    # the config int must not have leaked in
    assert 'in_features' not in names
    # it must be a generator, not a list
    import types
    assert isinstance(lin.get_children(), types.GeneratorType)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(3)

class MiniTensor:
    def __init__(self, data):
        self.data = data

class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            keep = isinstance(val, MiniTensor)
            if keep:
                yield name, val

class Linear(Module):
    def __init__(self):
        self.weight = MiniTensor(t.randn(4, 3))
        self.bias = MiniTensor(t.zeros(4))
        self.in_features = 3

print([n for n, _ in Linear().get_children()])
```
</details>